<a href="https://colab.research.google.com/github/asiahcrutchfield/ground-level-languages-ai/blob/main/gll_intro_to_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q allosaurus torchcrepe librosa soundfile scipy numpy

import os, json, zipfile
import numpy as np
import torch
import torchaudio
import torchcrepe
import librosa
import soundfile as sf
from scipy.signal import find_peaks
from allosaurus.app import read_recognizer

os.makedirs("audio/slices", exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Load Torchaudio MMS Forced Aligner (millisecond phoneme boundaries) + Allosaurus
bundle = torchaudio.pipelines.MMS_FA
fa_model = bundle.get_model().to(device)
fa_tokenizer = bundle.get_tokenizer()
fa_aligner = bundle.get_aligner()
recognizer = read_recognizer()

IPA_VOWELS = set("aeiouyɨʉɯuɪʏʊeøɘɵɤoəɛœɜɞʌɔæɐaɶɑɒɚɝ")

def classify_phone(ipa):
    return "vowel" if any(ch in IPA_VOWELS for ch in ipa) else "consonant"

def apply_fade(clip, sr=16000, fade_ms=10):
    fade_len = min(int(sr * (fade_ms / 1000.0)), len(clip) // 2)
    if fade_len <= 1:
        return clip
    win = np.ones(len(clip), dtype=np.float32)
    win[:fade_len] = np.linspace(0.0, 1.0, fade_len)
    win[-fade_len:] = np.linspace(1.0, 0.0, fade_len)
    return clip * win

def synthesize_humming_track(y, f0, periodicity, sr=16000, hop_length=160):
    total_samples = len(y)
    freq_interp = np.interp(np.arange(total_samples), np.linspace(0, total_samples, len(f0)), f0)
    conf_interp = np.interp(np.arange(total_samples), np.linspace(0, total_samples, len(periodicity)), periodicity)
    rms = librosa.feature.rms(y=y, frame_length=512, hop_length=hop_length)[0]
    rms_interp = np.interp(np.arange(total_samples), np.linspace(0, total_samples, len(rms)), rms)
    rms_norm = rms_interp / (np.max(rms_interp) + 1e-6)
    voiced_mask = (conf_interp > 0.15) & (rms_norm > 0.12)
    amp = np.where(voiced_mask, 0.35 * rms_norm, 0.0)
    phase = (2.0 * np.pi * np.cumsum(freq_interp)) / sr
    return (amp * (0.8 * np.sin(phase) + 0.2 * np.sin(2.0 * phase))).astype(np.float32)

def process_word(word_id, mp3_path, visual_path):
    y_raw, sr = librosa.load(mp3_path, sr=16000)
    y, _ = librosa.effects.trim(y_raw, top_db=28)
    y = np.pad(y, (int(0.05 * sr), int(0.05 * sr)), mode="constant")
    temp_wav = f"/content/{word_id}_16k.wav"
    sf.write(temp_wav, y, sr)

    # STEP 1: MMS_FA Forced Alignment (exact millisecond boundaries on real human speech)
    waveform = torch.tensor(y).unsqueeze(0).to(device)
    with torch.inference_mode():
        emission, _ = fa_model(waveform)
    tokens = fa_tokenizer([word_id])
    token_spans = fa_aligner(emission[0], tokens)[0]

    # Calculate ratio from emission frames to audio samples
    ratio = waveform.size(1) / emission.size(1)

    # Use Allosaurus to classify each aligned segment as consonant vs. vowel
    raw_phones = recognizer.recognize(temp_wav, lang_id='eng', timestamp=False, topk=1).split()

    phonemes_out = []
    for idx, span in enumerate(token_spans):
        # Add 15ms cushion so initial/final consonants retain natural acoustic release
        s_idx = max(0, int(span.start * ratio) - int(0.015 * sr))
        e_idx = min(len(y), int(span.end * ratio) + int(0.020 * sr))

        clip = apply_fade(y[s_idx:e_idx], sr=sr, fade_ms=10)
        clip_path = f"audio/slices/{word_id}_p{idx+1}.wav"
        sf.write(clip_path, clip, sr)

        # Match consonant/vowel category from Allosaurus or orthographic letter fallback
        ipa_hint = raw_phones[idx] if idx < len(raw_phones) else ("a" if word_id[idx] in "aeiou" else "k")
        phonemes_out.append({
            "id": f"{word_id}_p{idx+1}",
            "kind": classify_phone(ipa_hint),
            "audio_url": clip_path
        })

    print(f"[{word_id}] Forced-aligned {len(phonemes_out)} crisp human phoneme slices.")

    # STEP 2: Syllable segmentation
    rms = librosa.feature.rms(y=y, frame_length=512, hop_length=160)[0]
    peaks, _ = find_peaks(rms, distance=12, prominence=np.max(rms) * 0.2)
    if len(peaks) == 0:
        peaks = [len(rms) // 2]
    boundaries = [0] + [int((peaks[i] + peaks[i+1]) / 2) for i in range(len(peaks) - 1)] + [len(rms)]
    syllables_out = []
    for idx in range(len(boundaries) - 1):
        s_s, e_s = (boundaries[idx] * 160) / sr, (boundaries[idx+1] * 160) / sr
        dur = max(0.08, e_s - s_s)
        clip_path = f"audio/slices/{word_id}_s{idx+1}.wav"
        sf.write(clip_path, apply_fade(y[int(s_s * sr):int(e_s * sr)], sr=sr), sr)
        syllables_out.append({
            "id": f"{word_id}_s{idx+1}",
            "width_px": int(min(180, max(65, dur * 240))),
            "audio_url": clip_path
        })

    # STEP 3: TorchCREPE F0 humming
    pitch, periodicity = torchcrepe.predict(
        torch.tensor(y).unsqueeze(0), sr, hop_length=160, fmin=65.0, fmax=500.0,
        model="full", decoder=torchcrepe.decode.viterbi, return_periodicity=True, device=device
    )
    frequency = pitch.squeeze(0).cpu().numpy()
    conf = periodicity.squeeze(0).cpu().numpy()
    hum_out = f"audio/slices/{word_id}_hum.wav"
    sf.write(hum_out, synthesize_humming_track(y, frequency, conf, sr=sr), sr)

    tones_out = []
    for idx in range(len(syllables_out)):
        s_f = int((idx / len(syllables_out)) * len(frequency))
        e_f = int(((idx + 1) / len(syllables_out)) * len(frequency))
        slope = float(frequency[e_f-1] - frequency[s_f]) if e_f - s_f > 1 else 0.0
        style = "solid" if slope > 12 else ("ring" if slope < -12 else "dot")
        tones_out.append({"id": f"{word_id}_t{idx+1}", "style": style, "audio_url": hum_out})

    return {
        "id": word_id,
        "visual_url": visual_path,
        "full_audio_url": f"audio/{word_id}.mp3",
        "humming_audio_url": hum_out,
        "phonemes": phonemes_out,
        "syllables": syllables_out,
        "tones": tones_out
    }

words = [
    ("cat", "/content/cat.mp3", "images/visuals/cat.jpg"),
    ("dog", "/content/dog.mp3", "images/visuals/dog.jpg"),
    ("walk", "/content/walk.mp3", "images/visuals/walk.gif"),
    ("jump", "/content/jump.mp3", "images/visuals/jump.gif"),
    ("tree", "/content/tree.mp3", "images/visuals/tree.jpg")
]

dataset = {"items": [process_word(w, p, v) for w, p, v in words if os.path.exists(p)]}
with open("lesson_data.json", "w") as f:
    json.dump(dataset, f, indent=2)

with zipfile.ZipFile("gll_update1_assets.zip", "w") as zf:
    zf.write("lesson_data.json")
    for root, _, files in os.walk("audio/slices"):
        for file in files:
            zf.write(os.path.join(root, file))

print("Done! Extract gll_update1_assets.zip into 'Intro to AI Project/'.")

Downloading: "https://dl.fbaipublicfiles.com/mms/torchaudio/ctc_alignment_mling_uroman/model.pt" to /root/.cache/torch/hub/checkpoints/model.pt


100%|██████████| 1.18G/1.18G [00:20<00:00, 62.7MB/s]


Done! Extract gll_update1_assets.zip into 'Intro to AI Project/'.
